# AI fact-checking agent — V1: exact verified-claim cache

V1 reuses V0's rule-based checkability and NLI decisions, adds claim caching, and gives fresh retrieval
shorter timeouts. Inputs are still standalone English claims. Run from the repository root in the
Python 3.12 `.venv` kernel.

A valid exact claim-cache hit returns the stored verdict and citations. A miss verifies fresh evidence.
Supported/contradicted results may be cached even if an unrelated page failed to download.
Fatal search failures, uncertain results and missing citations remain ineligible.

Models load once. Original V0 remains available for comparison. The benchmark in this notebook applies
the same bounded-retrieval configuration to its uncached and cached runs.

## 1. Reuse the existing notebook definitions

Notebooks are not ordinary Python modules. The small helper below reads specific code cells by their IDs.
It loads V0's imports, model and functions, and the database's schema/helper definitions.

Each notebook gets its own dictionary of variables. We do not execute their benchmark, demo or database-reset
cells. This keeps V0 and the database notebook as the sources of truth without copying their implementations.
A missing cell ID raises an error, so a notebook reorganisation cannot silently load the wrong code.

In [ ]:
import hashlib
import json
import math
import re
import statistics
import tempfile
import time
from pathlib import Path


def load_cells(filename, cell_ids, namespace):
    notebook = json.loads(Path(filename).read_text())
    source_by_id = {cell["id"]: "".join(cell["source"]) for cell in notebook["cells"]}
    source = "\n\n".join(source_by_id[cell_id] for cell_id in cell_ids)
    exec(compile(source, filename, "exec"), namespace)
    return source


baseline = {}
baseline_source = load_cells("ai_agent.ipynb", [
    "baseline-02", "baseline-04", "baseline-06", "baseline-08", "baseline-10",
], baseline)
check_claim_v0 = baseline["check_claim"]

database = {}
load_cells("database.ipynb", [
    "83282e0a", "eec9e7b9", "5db5fa71", "facd8054", "c18214d0", "3509405e",
], database)
connect = database["connect"]
create_schema = database["create_schema"]
now_ms = database["now_ms"]

## 2. Use shorter retrieval limits
The original V0 notebook stays unchanged. A second copy of its decision function uses the helpers below,
with the same NLI model, relevance rule, thresholds and passage limit.

Search tries DuckDuckGo, then Brave only if the first fails or returns no results. Each provider has a
4-second request timeout, with at most two sequential attempts. If both fail, the error lists both causes.
Page downloads use 3-second socket timeouts, at most two redirects, a 1 MB size cap, and no automatic request retries. An 8-second page-stage
budget stops further downloads and is checked between body reads. All snippets are collected first.

These are request/socket limits and a best-effort stage budget, **not a guaranteed whole-request deadline**.
DNS resolution, redirects, a blocked socket read and HTML processing can overrun the checks.
Strict cross-platform cancellation would require a separate worker/process; it is not claimed here.

Requests is already installed through the pinned Transformers dependencies; no new package is required.
Cached and uncached comparison runs below use these SAME retrieval controls.
Original V0 measurements remain available in `ai_agent.ipynb`.

Bing text search is disabled in the pinned DDGS 9.16.0. Configured providers are checked against the
installed engine registry before requests, preventing DDGS from silently selecting automatic providers.
A successful fallback prints the failed first attempt and which backend actually returned results.
An enabled backend can still return no results; this does not establish that the claim is false.


In [ ]:
import requests
from ddgs.engines import ENGINES as DDGS_ENGINES

RETRIEVAL_SETTINGS = {
    "search_timeout_seconds": 4,
    "search_backends": ["duckduckgo", "brave"],
    "page_timeout_seconds": 3,
    "page_budget_seconds": 8,
    "max_download_bytes": 1_000_000,
    "max_redirects": 2,
}


def search_web_v1(claim):
    # The pinned DDGS silently switches to auto for disabled/unknown backends.
    # Check its installed registry before issuing any requests.
    unavailable = set(RETRIEVAL_SETTINGS["search_backends"]) - DDGS_ENGINES["text"].keys()
    if unavailable or not RETRIEVAL_SETTINGS["search_backends"]:
        raise ValueError(f"Configure enabled DDGS backends; unavailable: {sorted(unavailable)}")
    failures = []
    # Try the second provider only when the first fails or returns no results.
    for backend in RETRIEVAL_SETTINGS["search_backends"]:
        try:
            hits = list(baseline["DDGS"](
                timeout=RETRIEVAL_SETTINGS["search_timeout_seconds"],
            ).text(claim, max_results=baseline["MAX_RESULTS"], backend=backend))
            if hits:
                if failures:
                    print("Search fallback:", "; ".join(failures),
                          f"-> {backend}: {len(hits)} results")
                return hits
            failures.append(f"{backend}: no results")
        except Exception as error:
            failures.append(f"{backend}: {type(error).__name__}: {error}")
    raise RuntimeError("; ".join(failures))


def download_page_v1(url, session, deadline):
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        raise TimeoutError("Page download budget exhausted.")
    timeout = min(RETRIEVAL_SETTINGS["page_timeout_seconds"], remaining)
    with session.get(url, timeout=(timeout, timeout), stream=True) as response:
        response.raise_for_status()
        chunks, size = [], 0
        while True:
            if time.perf_counter() >= deadline:
                raise TimeoutError("Page download budget exhausted.")
            # read1 returns available data, rather than waiting to fill a large chunk.
            chunk = response.raw.read1(64 * 1024, decode_content=True)
            if not chunk:
                break
            size += len(chunk)
            if size > RETRIEVAL_SETTINGS["max_download_bytes"]:
                raise ValueError("Page exceeds the download size limit.")
            chunks.append(chunk)
    return b"".join(chunks)


def collect_evidence_v1(claim, search_results):
    claim_words = baseline["content_words"](claim)
    passages, warnings, seen = [], [], set()

    def add_passage(text, url, source_type):
        text = re.sub(r"\s+", " ", text).strip()[:700]
        relevance = len(claim_words & baseline["content_words"](text)) / max(len(claim_words), 1)
        if text and text.lower() not in seen and relevance >= 0.5:
            seen.add(text.lower())
            passages.append({
                "text": text, "url": url, "source_type": source_type,
                "overlap": len(claim_words & baseline["content_words"](text)),
                "relevance": round(relevance, 3),
            })

    # Collect every available snippet before a slow page can consume the budget.
    hits = search_results[:baseline["MAX_RESULTS"]]
    for hit in hits:
        url = hit.get("href") or hit.get("url")
        snippet = hit.get("body") or hit.get("snippet") or ""
        if url and snippet:
            add_passage(snippet, url, "search_snippet")

    deadline = time.perf_counter() + RETRIEVAL_SETTINGS["page_budget_seconds"]
    with requests.Session() as session:
        # Requests' default adapters do not retry failed downloads.
        session.max_redirects = RETRIEVAL_SETTINGS["max_redirects"]
        session.headers["User-Agent"] = "hes-hackathon/0.1 (student fact-checking prototype)"
        for hit in hits:
            url = hit.get("href") or hit.get("url")
            if not url:
                continue
            if time.perf_counter() >= deadline:
                warnings.append("Page budget exhausted; remaining downloads skipped.")
                break
            try:
                page_deadline = min(
                    deadline, time.perf_counter() + RETRIEVAL_SETTINGS["page_timeout_seconds"]
                )
                html = download_page_v1(url, session, page_deadline)
                text = baseline["extract"](html, include_comments=False) if html else None
                if not text:
                    warnings.append(f"{url}: no readable text")
                    continue
                sentences = re.split(r"(?<=[.!?])\s+", re.sub(r"\s+", " ", text))
                for sentence in sentences:
                    add_passage(sentence, url, "page")
            except Exception as error:
                warnings.append(f"{url}: {type(error).__name__}: {error}")
    passages.sort(
        key=lambda passage: (passage["relevance"], passage["source_type"] == "page"),
        reverse=True,
    )
    return passages[:baseline["MAX_PASSAGES"]], warnings


# Reuse the same V0 decision function in an isolated dictionary with the new retrieval helpers.
bounded_verifier = dict(
    baseline, search_web=search_web_v1, collect_evidence=collect_evidence_v1,
)
load_cells("ai_agent.ipynb", ["baseline-10"], bounded_verifier)
check_claim_bounded = bounded_verifier["check_claim"]

## 3. Open the V1 cache

The existing claim/evidence/link tables are reused in `database/v1_claim_cache.db`.
One diagnostic `warnings_json` column is added to `verified_claims` if absent.
No cache reset or manual database deletion is needed.

The one-hour TTL and 1,000-entry capacity are experiment settings. Expired/least-recently-used claims
and their unshared citation rows are pruned. V1 stores passages for citations; it does not search them
as an evidence/RAG cache.

The verifier fingerprint includes V0 code, model revision, retrieval settings and the revised cache policy.
Old entries from the earlier policy are not reused. Semantic metadata fields are not inferred by V1.

In [ ]:
CACHE_DB_PATH = "database/v1_claim_cache.db"
CACHE_TTL_SECONDS = 60 * 60
MAX_CACHE_ENTRIES = 1000

if CACHE_TTL_SECONDS <= 0 or MAX_CACHE_ENTRIES < 1:
    raise ValueError("Use a positive cache TTL and capacity.")

model_revision = getattr(baseline["model"].config, "_commit_hash", None) or "unknown"
fingerprint = baseline_source + json.dumps(RETRIEVAL_SETTINGS, sort_keys=True) + "cache-policy-r2"
code_hash = hashlib.sha256(fingerprint.encode()).hexdigest()[:16]
VERIFIER_VERSION = (
    f"v1-exact:r2:{baseline['MODEL_NAME']}@{model_revision}:{code_hash}"
)


def open_cache(path):
    connection = connect(path)
    create_schema(connection)
    columns = {row[1] for row in connection.execute("PRAGMA table_info(verified_claims)")}
    if "warnings_json" not in columns:
        with connection:
            connection.execute(
                "ALTER TABLE verified_claims ADD COLUMN warnings_json TEXT NOT NULL DEFAULT '[]'"
            )
    return connection


if "cache_con" in globals():
    cache_con.close()
cache_con = open_cache(CACHE_DB_PATH)
print("Cache:", CACHE_DB_PATH)
print("Verifier version:", VERIFIER_VERSION)

## 4. Read and write complete cached verdicts

Matching only collapses whitespace. Case, punctuation, negation and quantities remain part of the key.
An unexpired hit reads the verdict, citations, original NLI scores and original verification warnings.

Cache supported/contradicted results when they have citations and no **fatal search error**.
A failed page becomes a warning; a valid search snippet from another page can still support a verdict.
Uncertain, non-checkable and uncited results remain ineligible.

Warnings saved with a cache entry describe the original verification, not a new download attempt.
A hit updates its counter and last-use time, but does not search the web or rerun NLI.
The short assessment describes the verdict; it is not a generated reasoning trace.

In [ ]:
def normalise_claim(text):
    return re.sub(r"\s+", " ", text).strip()


def delete_cached_claims(connection, claim_ids):
    for claim_id in claim_ids:
        evidence_ids = connection.execute(
            "SELECT evidence_id FROM verdict_evidence WHERE verified_claim_id = ?",
            (claim_id,),
        ).fetchall()
        connection.execute("DELETE FROM verified_claims WHERE id = ?", (claim_id,))
        for (evidence_id,) in evidence_ids:
            connection.execute(
                """DELETE FROM evidence WHERE id = ?
                   AND NOT EXISTS (
                     SELECT 1 FROM verdict_evidence WHERE evidence_id = evidence.id
                   )""",
                (evidence_id,),
            )


def prune_cache(connection):
    expired = connection.execute(
        """SELECT id FROM verified_claims
           WHERE model_version LIKE 'v1-exact:%' AND expires_at <= ?""",
        (now_ms(),),
    ).fetchall()
    delete_cached_claims(connection, [row[0] for row in expired])
    overflow = connection.execute(
        """SELECT id FROM verified_claims
           WHERE model_version LIKE 'v1-exact:%'
           ORDER BY COALESCE(last_used_at, verified_at) DESC, id DESC
           LIMIT -1 OFFSET ?""",
        (MAX_CACHE_ENTRIES,),
    ).fetchall()
    delete_cached_claims(connection, [row[0] for row in overflow])


def lookup_claim(text, connection):
    row = connection.execute(
        """SELECT id, verdict, explanation, verified_at, expires_at, warnings_json
           FROM verified_claims
           WHERE claim_text = ? AND model_version = ? AND expires_at > ?
           ORDER BY verified_at DESC, id DESC LIMIT 1""",
        (normalise_claim(text), VERIFIER_VERSION, now_ms()),
    ).fetchone()
    if row is None:
        return None

    claim_id, verdict, explanation, verified_at, expires_at, warnings_json = row
    passages = connection.execute(
        """SELECT e.evidence_text, e.source_url, ve.scores_json
           FROM verdict_evidence AS ve
           JOIN evidence AS e ON e.id = ve.evidence_id
           WHERE ve.verified_claim_id = ? ORDER BY e.id""",
        (claim_id,),
    ).fetchall()
    if not passages:
        return None

    evidence = [
        {"text": passage, "url": url, **json.loads(metadata or "{}")}
        for passage, url, metadata in passages
    ]
    with connection:
        connection.execute(
            """UPDATE verified_claims
               SET last_used_at = ?, hit_count = hit_count + 1 WHERE id = ?""",
            (now_ms(), claim_id),
        )
    return {
        "claim": text, "verdict": verdict, "evidence": evidence, "errors": [],
        "warnings": json.loads(warnings_json or "[]"),
        "explanation": explanation, "verified_claim_id": claim_id,
        "verified_at": verified_at, "expires_at": expires_at,
    }


def save_claim(text, result, connection):
    if result["verdict"] not in {"supported", "contradicted"}:
        return None
    if result["errors"] or not result["evidence"]:
        return None
    if not all(passage.get("text") and passage.get("url") for passage in result["evidence"]):
        return None

    verified_at = now_ms()
    expires_at = verified_at + CACHE_TTL_SECONDS * 1000
    stance = "supports" if result["verdict"] == "supported" else "contradicts"
    with connection:
        cursor = connection.execute(
            """INSERT INTO verified_claims
               (claim_text, verdict, explanation, verified_at, expires_at,
                model_version, last_used_at, warnings_json)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (normalise_claim(text), result["verdict"], result["explanation"],
             verified_at, expires_at, VERIFIER_VERSION, verified_at,
             json.dumps(result.get("warnings", []))),
        )
        claim_id = cursor.lastrowid
        for passage in result["evidence"]:
            evidence_id = connection.execute(
                """INSERT INTO evidence (evidence_text, source_url, fetched_at)
                   VALUES (?, ?, ?)""",
                (passage["text"], passage["url"], verified_at),
            ).lastrowid
            metadata = {k: v for k, v in passage.items() if k not in {"text", "url"}}
            connection.execute(
                """INSERT INTO verdict_evidence
                   (verified_claim_id, evidence_id, stance, scores_json)
                   VALUES (?, ?, ?, ?)""",
                (claim_id, evidence_id, stance, json.dumps(metadata)),
            )
        prune_cache(connection)
    return {"verified_claim_id": claim_id, "verified_at": verified_at,
            "expires_at": expires_at}

## 5. Put the cache around the verifier

A miss runs the V0 decision function with bounded retrieval.
Its search failures stay in `errors`; page failures and budget messages move to `warnings`.
Only fatal errors block an otherwise eligible cached verdict.

A hit measures lookup/citation-read/commit time. Its scores and warnings belong to the original
verification, and its timings describe the current cache request.

In [ ]:
def check_claim_v1(text, connection=None):
    connection = cache_con if connection is None else connection
    started = time.perf_counter()

    lookup_started = time.perf_counter()
    result = lookup_claim(text, connection)
    lookup_ms = (time.perf_counter() - lookup_started) * 1000
    if result is not None:
        result.update(cache_hit=True, path="claims_cache")
        result["timings_ms"] = {
            "cache_lookup": round(lookup_ms, 2),
            "total": round((time.perf_counter() - started) * 1000, 2),
        }
        return result

    result = check_claim_bounded(text)
    # V0 uses one error list; distinguish search failure from recoverable page warnings.
    result["warnings"] = [
        error for error in result["errors"] if not error.startswith("Search failed:")
    ]
    result["errors"] = [
        error for error in result["errors"] if error.startswith("Search failed:")
    ]
    result["cache_hit"] = False
    result["path"] = None if result["verdict"] == "not_checkable" else "fresh"
    result["explanation"] = {
        "supported": "The retrieved evidence supports this claim.",
        "contradicted": "The retrieved evidence contradicts this claim.",
        "uncertain": "The retrieved evidence did not produce a clear verdict.",
        "not_checkable": "The V0 rules did not classify this input as checkable.",
    }[result["verdict"]]

    write_started = time.perf_counter()
    cache_record = save_claim(text, result, connection)
    write_ms = (time.perf_counter() - write_started) * 1000
    if cache_record:
        result.update(cache_record)
    result["timings_ms"]["cache_lookup"] = round(lookup_ms, 2)
    result["timings_ms"]["cache_write"] = round(write_ms, 2)
    result["timings_ms"]["total"] = round((time.perf_counter() - started) * 1000, 2)
    return result

## 6. Try a claim twice

A supported/contradicted result with citations should be saved even if another page produces a warning.
Its repeat should use `claims_cache`. An already-populated cache may make both requests hits.
An uncertain result or fatal search failure remains fresh.

Verification warnings shown on a hit are preserved from the original verification.

In [ ]:
first = check_claim_v1("Switzerland has 26 cantons.")
second = check_claim_v1("Switzerland has 26 cantons.")

for name, result in [("First", first), ("Second", second)]:
    print(name, result["verdict"], "|", result["path"], "|",
          result["timings_ms"]["total"], "ms")
    if result["errors"]:
        print("Search errors:", result["errors"])
    if result.get("warnings"):
        print("Verification warnings:", result["warnings"])

print(json.dumps(second, indent=2))

## 7. Compare uncached, V1 cold and V1 warm

All three passes use the SAME bounded search/download settings and the same V0 NLI decisions.
The first pass bypasses caching. A fresh temporary database makes the next pass cold, and the third
reuses that database. The persistent cache and application data are not cleared.

Each pass still performs independent live searches on misses, so verdicts and timings may vary.
Keep original V0's unrestricted retrieval measurements separate from this comparison.

Report all factual inputs, cache hits alone, and misses alone. This keeps a slow warm-pass miss visible
without presenting it as cache-hit latency. Empty groups show null timings.
Uncertain/search-failure cases remain misses. With three factual inputs this is only a smoke benchmark.

Search failures print their provider errors and mark the pass `incomplete_search_failures`.
All-request timings include failures; `cited_verdicts_only` measures only supported/contradicted results
with citations and no search error. `cited_verdict_under_1_second_percent` uses ALL factual inputs as its
denominator, so fast failures cannot improve that rate. These metrics do not imply verdict correctness;
accuracy is still reported separately. The benchmark is live, not a replay of saved evidence.

The Switzerland example in section 6 uses a DIFFERENT database from this benchmark. A hit there does
not preload the benchmark. Warm-pass claims only hit when the cold pass saved a decisive cited verdict.


In [ ]:
examples = [
    ("Switzerland has 26 cantons.", "supported"),
    ("Paris is the capital of Germany.", "contradicted"),
    ("The Moon is a natural satellite of Earth.", "supported"),
    ("I think Switzerland is beautiful.", "not_checkable"),
    ("What is the capital of France?", "not_checkable"),
]

benchmark_runs = {}
with tempfile.TemporaryDirectory(prefix="veriflow-v1-") as folder:
    benchmark_con = open_cache(str(Path(folder) / "cache.db"))
    try:
        for label in ["Uncached (same retrieval)", "V1 cold", "V1 warm"]:
            results = []
            for text, expected in examples:
                if label == "Uncached (same retrieval)":
                    result = check_claim_bounded(text)
                else:
                    result = check_claim_v1(text, benchmark_con)
                result["expected"] = expected
                results.append(result)
                print(label, "|", result["verdict"], "|",
                      result.get("path", "fresh"), "|",
                      result["timings_ms"]["total"], "ms |", text)
                for error in result["errors"]:
                    if error.startswith("Search failed:"):
                        print("  ", error)
            benchmark_runs[label] = results
    finally:
        benchmark_con.close()


def latency_summary(results):
    values = sorted(result["timings_ms"]["total"] for result in results)
    return {
        "count": len(values),
        "p50_ms": statistics.median(values) if values else None,
        "p95_ms": values[math.ceil(0.95 * len(values)) - 1] if values else None,
        "under_1_second_percent": 100 * sum(ms < 1000 for ms in values) / len(values) if values else None,
    }


def summarise_run(results):
    factual = [r for r in results if r["expected"] != "not_checkable"]
    hits = [r for r in factual if r.get("cache_hit", False)]
    misses = [r for r in factual if not r.get("cache_hit", False)]
    search_failures = [
        {"claim": r["claim"], "errors": [e for e in r["errors"] if e.startswith("Search failed:")]}
        for r in factual if any(e.startswith("Search failed:") for e in r["errors"])
    ]
    cited = [r for r in factual if r["verdict"] in {"supported", "contradicted"}
             and r.get("evidence")
             and not any(e.startswith("Search failed:") for e in r["errors"])]
    return {
        "status": "incomplete_search_failures" if search_failures else "completed",
        "all_factual": latency_summary(factual),
        "cited_verdicts_only": latency_summary(cited),
        "cited_verdict_under_1_second_percent": (
            100 * sum(r["timings_ms"]["total"] < 1000 for r in cited) / len(factual)
            if factual else None
        ),
        "cache_hits_only": latency_summary(hits),
        "cache_misses_only": latency_summary(misses),
        "claim_cache_hit_percent": 100 * len(hits) / len(factual) if factual else None,
        "verdict_accuracy": statistics.mean(r["verdict"] == r["expected"] for r in factual) if factual else None,
        "inputs_with_search_errors": len(search_failures),
        "search_errors": search_failures,
        "inputs_with_verification_warnings": sum(
            bool(r.get("warnings")) or any(not e.startswith("Search failed:") for e in r["errors"])
            for r in factual
        ),
    }

summary = {
    "retrieval_mode": "live_web_bounded",
    "retrieval_settings": RETRIEVAL_SETTINGS,
    "model_startup_ms": baseline["startup_ms"],
    "verifier_version": VERIFIER_VERSION,
    "cache_ttl_seconds": CACHE_TTL_SECONDS,
    "cache_capacity": MAX_CACHE_ENTRIES,
    "runs": {label: summarise_run(results) for label, results in benchmark_runs.items()},
}
print(json.dumps(summary, indent=2))

## 8. Inspect the persistent cache

Section 6 uses the persistent file; the benchmark uses a separate temporary file.
Only entries for the current verifier fingerprint are shown.

In [ ]:
rows = cache_con.execute(
    """SELECT id, claim_text, verdict, hit_count, verified_at, expires_at
       FROM verified_claims WHERE model_version = ?
       ORDER BY last_used_at DESC LIMIT 10""",
    (VERIFIER_VERSION,),
).fetchall()
for row in rows:
    print(row)

# Optional when you have finished testing:
# cache_con.close()

## What this version establishes

A valid hit skips retrieval and NLI while retaining verdicts, citations, original scores and original warnings.
Page failures no longer prevent caching a clear cited result. Search failures and uncertain results remain misses.

The benchmark separates hit/miss latency and controls retrieval settings across its three passes.
Original V0's unrestricted timings must be reported separately, because the retrieval limits changed here.

V0's detection and verification limitations remain: caching cannot repair an incorrect verdict or establish
source trustworthiness. This notebook does not split compounds or resolve references.
The separate Qwen extraction experiments are not part of V1.

The page budget is cooperative, not an absolute cancellation guarantee. For a strict wall-clock limit and
later concurrency integration, run retrieval in separately managed workers. This version remains sequential.

Restart the kernel and run all cells after this update. The diagnostic-column migration and new fingerprint
are automatic; no manual database reset is required.